# 🚀 Модуль 2. Урок 2. Агрегация данных, GROUP BY и разница с HAVING

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_02_aggregations_group_by_having.ipynb)

## 📌 Бизнес-контекст и методология
Агрегатные функции (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`) сжимают строки в бизнес-метрики. `WHERE` фильтрует до агрегации, `HAVING` — после.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

db_path = ensure_dataset("ecommerce.db")
conn = sqlite3.connect(db_path)
print(f"🟢 Подключение к базе данных успешно установлено: {db_path}")


🟢 Подключение к базе данных успешно установлено: course\datasets\ecommerce.db


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(ensure_dataset("ecommerce.db"))

query = """
SELECT 
    c.city,
    COUNT(DISTINCT c.customer_id) AS total_customers,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.total_amount), 2) AS total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'Completed'
GROUP BY c.city
HAVING total_orders >= 10
ORDER BY total_revenue DESC;
"""

df_cities = pd.read_sql_query(query, conn)
df_cities


,city,total_customers,total_orders,total_revenue
0,Samara,146,291,404842.27
1,Kazan,136,310,398060.52
2,Nizhny Novgorod,128,308,396501.57
3,Yekaterinburg,132,289,390864.79
4,Saint Petersburg,128,262,354896.68
5,Moscow,121,286,343901.98
6,Novosibirsk,125,260,321291.80


### 🎯 Практическое задание 1: Поиск постоянных клиентов (Repeat Buyers)
Найдите клиентов с более чем 2 завершенными заказами.

In [3]:
### ВАШ КОД ЗДЕСЬ
query_repeats = """
SELECT 
    o.customer_id,
    COUNT(DISTINCT o.order_id) AS orders_count,
    ROUND(SUM(oi.total_amount), 2) AS customer_ltv
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'Completed'
GROUP BY o.customer_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY customer_ltv DESC;
"""

df_repeats = pd.read_sql_query(query_repeats, conn)
print(f"Постоянных клиентов: {len(df_repeats)}")
print(df_repeats.head(5))

assert not df_repeats.empty, "Список не должен быть пустым"
assert (df_repeats['orders_count'] >= 2).all(), "Заказов должно быть не менее 2"
print("✅ Тест успешно пройден: когорта постоянных клиентов собрана!")


Постоянных клиентов: 470
   customer_id  orders_count  customer_ltv
0           24            11      19656.48
1          869            11      15341.35
2          116            10      14842.05
3          517             7      14698.81
4          581             8      14267.41
✅ Тест успешно пройден: когорта постоянных клиентов собрана!


### 💼 COUNT(*) vs COUNT(column)
`COUNT(*)` считает все строки, `COUNT(DISTINCT col)` считает уникальные сущности.